In [1]:
import numpy as np
import finesse
import matplotlib.pyplot as plt
from utils.finesse_base import base_kat
from finesse.knm import Map
from finesse.utilities.maps import circular_aperture
from utils.sim import finesse_sim

/home/xuesi.ma/.conda/envs/ligoopt/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Finesse Testing Ground

In [11]:
kat = base_kat.deepcopy()

ITM_ROC = -1934
ETM_ROC = 2245
initial_guess = -1
ETM_ROC = ETM_ROC + ETM_ROC*.2
ITM_ROC = ITM_ROC + ITM_ROC*0
print(f"ITM_ROC: {ITM_ROC}")
print(f"ETM_ROC: {ETM_ROC}")

kat.ITM.Rc = ITM_ROC
kat.ETM.Rc = ETM_ROC
x = y = np.linspace(-0.17, 0.17, 100)

kat.ETM.surface_map = Map(x, y, amplitude=circular_aperture(x,y,0.17))
# kat.ETM.phi = initial_guess

out = kat.run("run_locks(display_progress=true,pre_step=print_model_attr(ETM.phi))")


print(f"\nThe ETM tunning AFTER the successful lock is {kat.ETM.phi.value} deg")
print("g-value is ", kat.cavArm.g[0])


ITM_ROC: -1934
ETM_ROC: 2694.0
Error Signal Residuals at Each Iteration (W):
                         lock_length  
Iteration Number   0   ETM.phi=0.0
    2.64e-12   
Iteration Number   1   ETM.phi=-2.8129236915395315e-12
    1.80e-12   
Iteration Number   2   ETM.phi=-4.734640511448609e-12
    1.23e-12   
Iteration Number   3   ETM.phi=-6.047510026301384e-12
    8.40e-13   
The ETM tunning AFTER the successful lock is -6.047510026301384e-12 deg
g-value is  0.982311605935746


In [3]:
sol = kat.run( """
    series(
        eigenmodes(cavArm, 0,       name="c0"), 
    )
    """)
gamma_list = abs(sol["c0"].eigvalues)
total_loss = 0
for gamma in gamma_list:
    total_loss += 1 - gamma**2 - 0.014
print(total_loss)


0.0004016893280065093


In [2]:
kat_ideal = base_kat.deepcopy()
print("g-value is ", kat_ideal.cavArm.g[0])
# base_kat_g = base_kat.optical_network
# for node in base_kat_g.nodes():
#     name = node.replace('.', '_')
   
ref_power = kat_ideal.run("""Series(
                          run_locks(display_progress=true,pre_step=print_model_attr(ETM.phi)),
                          noxaxis(),
                          )""")
print("\nreference power:", ref_power["noxaxis"]["p_ETM_p1_i"])
kat_aperture = base_kat.deepcopy()
# kat_aperture.ETM.surface_map = Map(x, y, amplitude=circular_aperture(x,y,0.17))
# aperture_power = kat_aperture.run("run_locks(display_progress=true,pre_step=print_model_attr(ETM.phi))")["circ"]
# print(aperture_power)
# print((1 - (aperture_power/ref_power))*100)



g-value is  0.8302329780069692
Error Signal Residuals at Each Iteration (W):
                         lock_length  
Iteration Number   0   ETM.phi=0.0
    0.00e+00   
reference power: 282.0955214139097


In [5]:
86.86045632011474

86.86045632011474

In [ ]:
from perturbation import calc_perturbed_params
ETM_ROC_perturbed, ITM_ROC_perturbed, Cav_L_perturbed = calc_perturbed_params(999, 999, 4999)
print(f"ETM_ROC: {ETM_ROC_perturbed}")
print(f"ITM_ROC: {ITM_ROC_perturbed}")
print(f"Cav_L: {Cav_L_perturbed}")
results = []
for ETM_ROC_p, ITM_ROC_p, Cav_L_p in zip(ETM_ROC_perturbed, ITM_ROC_perturbed, Cav_L_perturbed):
        result = [ETM_ROC_p, ITM_ROC_p, Cav_L_p]
        results.append(result)
print(results)

ETM_ROC: (1002.2967, 999, 995.7033)
ITM_ROC: (1002.2967, 999, 995.7033)
Cav_L: (4999.001, 4999, 4998.999)
[[1002.2967, 1002.2967, 4999.001], [999, 999, 4999], [995.7033, 995.7033, 4998.999]]


In [3]:
ITM_ROC = -20
ETM_ROC = 18
power = finesse_sim((ITM_ROC, ETM_ROC))
pd_names = []
base_kat_g = base_kat.optical_network
for node in base_kat_g.nodes():
        name = node.replace('.', '_')
        pd_names.append(f"p_{name}")
powers = []
for name in pd_names:
    powers.append(power[name])
print(len(powers))

14


In [7]:
test_kat = base_kat.deepcopy()
test_kat.parse(f"bp q_value ITM.p1.i prop=q")
result = test_kat.run()
q_value = result["q_value"]
print(q_value)

(-1834.203538874628+427.8399492372375j)


In [9]:
test_change_kat = base_kat.deepcopy()
test_change_kat.ITM.Rc = -1934 - 20
test_change_kat.ETM.Rc = 2245 - 20
test_change_kat.parse(f"bp q_value_changed ITM.p1.i prop=q")
test_change_kat.parse(f"gauss fixed_gauss ITM.p1.i q={q_value}")
out = test_change_kat.run("""Series(
                                run_locks(max_iterations=100000,),
                                noxaxis(),
                                )""")["noxaxis"]
print(out["q_value_changed"])


(-1834.203538874628+427.83994923723753j)
